# Demo 8 Setup

This notebook creates the schema and tables used in **Demo 8: Creating and Querying the Different Types of Views**.

In [0]:
# Get current user and set up schema name
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = f"demo_views_{clean_username}"

# Create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}` COMMENT 'Demo schema for views'")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

print(f"\u2713 Schema ready: {catalog_name}.{schema_name}")

In [0]:
# Create raw_orders table - simulates messy source data that views will transform
import random
from pyspark.sql import Row
from datetime import date, timedelta

random.seed(55)

regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Food & Beverage', 'Sports']
statuses = ['completed', 'completed', 'completed', 'completed', 'pending', 'cancelled']  # weighted toward completed

rows = []
for i in range(500):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 89))
    region = random.choice(regions)
    category = random.choice(categories)
    status = random.choice(statuses)
    quantity = random.randint(1, 10)
    unit_price = round(random.uniform(10, 200), 2)
    discount_pct = random.choice([0, 0, 0, 5, 10, 15, 20])  # most orders have no discount
    
    rows.append(Row(
        order_id=1000 + i,
        order_date=order_date,
        region=region,
        product_category=category,
        status=status,
        quantity=quantity,
        unit_price=unit_price,
        discount_pct=discount_pct,
        customer_email=f"customer_{random.randint(100,999)}@example.com"
    ))

df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable("raw_orders")

count = spark.sql("SELECT COUNT(*) FROM raw_orders").first()[0]
print(f"\u2713 Table created: raw_orders ({count} rows) - source data for view demos")

In [0]:
# Drop any views left from a prior run (clean slate)
for view_name in ['v_completed_orders', 'v_daily_revenue_summary']:
    spark.sql(f"DROP VIEW IF EXISTS {view_name}")

# Materialized views may not be supported on all compute types
try:
    spark.sql("DROP MATERIALIZED VIEW IF EXISTS mv_regional_summary")
except Exception:
    pass  # MV not supported on this compute - that's fine

print("\u2713 Cleaned up any prior views")

In [0]:
print("")
print("="*60)
print("\u2713 SETUP COMPLETE")
print("="*60)
print(f"")
print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"")
print(f"Tables created:")
print(f"  \u2022 raw_orders - 500 orders with region, category, status, pricing")
print(f"")
print(f"This raw data simulates messy source data.")
print(f"In the demo, we'll create views to transform it into clean, reusable outputs.")